# सिनैप्टिक रूटिंग आर्किटेक्चर (एसआरए)
## 10: प्रैक्टिकल प्लगइन हॉट-स्वैप (जीरो-शॉट हॉट-स्वैप)

यह नोटबुक परम मॉड्यूलर लर्निंग को प्रदर्शित करता है जो एसआरए का असली सार है: ``भौतिक रूप से संयोजन (हॉट-स्वैप) कई विशेष प्लग-इन (सिनैप्स) जो स्वतंत्र रूप से और तथ्य के बाद बेस मॉडल में समानांतर में प्रशिक्षित होते हैं।''

### प्रदर्शन परिदृश्य
1. **बेस मॉडल लर्निंग**: सामान्य टेक्स्ट (`टेक्स्ट`) डेटा का उपयोग करके बुनियादी मॉडल सीखें।
2. **प्लग-इन ए की स्वतंत्र शिक्षा**: बेस मॉडल की प्रतिलिपि बनाएँ और सीखने के लिए प्रोग्रामिंग कोड (`कोड`) के लिए समर्पित एक सिनैप्स जोड़ें।
3. **प्लग-इन बी की स्वतंत्र शिक्षा**: बेस मॉडल की प्रतिलिपि बनाएँ, गणित (`गणित`) के लिए समर्पित एक सिनेप्स जोड़ें, और इसे स्वतंत्र रूप से और समानांतर में प्रशिक्षित करें।
4. **टेंसर भौतिक संयोजन (हॉट-स्वैप)**: उत्पादन वातावरण में बेस मॉडल की मेमोरी पर सीधे प्लगइन ए और प्लगइन बी के वेट टेंसर को कॉपी करें और उन्हें मर्ज करें।
5. **शून्य-शॉट पूर्ण अलगाव मूल्यांकन**: मेटाडेटा-संचालित रूटिंग मास्क फ़ंक्शन का उपयोग करते हुए, **बिना किसी पुनः सीख के**, हम साबित करते हैं कि सभी डोमेन के नुकसान दशमलव बिंदु तक स्वतंत्र सीखने से पूरी तरह मेल खाते हैं (कैटास्ट्रॉफिक फ़ॉरगेटिंग गणितीय रूप से शून्य है)।

## 0. पर्यावरण सेटअप (Google Colab के लिए)
यदि Google Colab पर चल रहा है, तो रिपॉजिटरी और आवश्यक लाइब्रेरी डाउनलोड करने के लिए नीचे दिए गए सेल को चलाएँ।

In [ ]:
# Colab環境の場合のみ実行
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch

# パスの追加
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')


In [ ]:
import sys
import os
import copy
import random
import torch
import torch.nn.functional as F
import tiktoken

# 開発中のSRAライブラリへのパスを通す

from sra_language_models import MoESRALanguageModel
from sra_experiment import make_optimizer

# 乱数シードの固定（再現性確保）
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

### 1. डेटासेट तैयार करना और बैच जनरेशन फ़ंक्शन
इस बार, हम तीन डोमेन का उपयोग करेंगे: `टेक्स्ट` (सामान्य टेक्स्ट), `कोड` (प्रोग्राम), और `गणित` (सूत्र)।

In [ ]:
tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

# データの読み込み
data = {}
for domain in ["code", "math", "text"]:
    path = f"../data/lang/{domain}.txt"
    if not os.path.exists(path):
        print(f"Warning: {path} not found. Creating a dummy dataset.")
        # create dummy data if it does not exist (for CI/CD environments)
        os.makedirs("../data/lang", exist_ok=True)
        with open(path, "w") as f:
            f.write(f"This is some dummy data for the {domain} domain.\n" * 100)
    
    with open(path, "r", encoding="utf-8") as f:
        text_content = (f.read() + "\n") * 5
    
    tokens = tokenizer.encode(text_content, allowed_special="all")
    data[domain] = torch.tensor(tokens, dtype=torch.long)

def get_batch(data_dict, batch_size, seq_len, domain):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    d_data = data_dict[domain]
    max_start = len(d_data) - seq_len - 1
    for i in range(batch_size):
        start = random.randint(0, max(0, max_start))
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
    return x, y

def eval_domain(model, domain, allowed_mask=None, eval_batches=5):
    # 評価時の再現性を担保するためにシードを固定
    set_seed(999)
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for _ in range(eval_batches):
            x, y = get_batch(data, 32, max_seq_len, domain)
            x, y = x.to(device), y.to(device)
            logits, _ = model(x, allowed_synapses_mask=allowed_mask)
            loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
            total_loss += loss.item()
    return total_loss / eval_batches

### 2. बेस मॉडल (टेक्स्ट डोमेन) का प्रशिक्षण
सबसे पहले, हम एक बुनियादी मॉडल (16 सिनैप्स) बनाएंगे जो केवल सामान्य पाठ सीखता है।

In [ ]:
# モデルハイパーパラメータ (少し規模を大きく設定)
dim = 128
layers = 4
num_base_synapses = 16
k = 2
syn_hidden = 256
max_seq_len = 64
device = "cuda" if torch.cuda.is_available() else "cpu"

base_model = MoESRALanguageModel(
    vocab_size, dim, layers, num_base_synapses, k, syn_hidden, max_seq_len=max_seq_len
).to(device)

opt_base = make_optimizer(base_model, 5e-4)

print("=== Training Base Model (TEXT) ===")
base_model.train()
for step in range(1, 301):
    x, y = get_batch(data, 32, max_seq_len, "text")
    x, y = x.to(device), y.to(device)
    logits, _ = base_model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    
    opt_base.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(base_model.parameters(), 1.0)
    opt_base.step()
    
    if step % 100 == 0:
        print(f"Step {step} | TEXT Loss: {loss.item():.4f}")

# ベースモデルの評価
loss_text_base = eval_domain(base_model, "text")
print(f"\nBase Model TEXT Evaluation Loss: {loss_text_base:.4f}")

### 3. प्लगइन ए की स्वतंत्र शिक्षा (कोड के लिए)
मान लें कि टीम ए बेस मॉडल की प्रतिलिपि बनाती है, प्रोग्रामिंग डेटा सीखती है, और एक प्लगइन बनाती है। आधार ज्ञान को पूरी तरह सुरक्षित रखने के लिए `freeze_base=True` सेट करें।

In [ ]:
print("=== Training Plugin A (CODE) independently ===")
plugin_a_model = copy.deepcopy(base_model)

# プラグイン用にシナプスを4つ追加し、既存の重みを完全凍結
plugin_a_model.add_synapses(4, freeze_base=True)

# 最適化器は新規シナプス（requires_grad=Trueのパラメータ）のみを更新
opt_a = make_optimizer(plugin_a_model, 5e-4)

# CODEドメインで新しいシナプスのみを使用するためのマスクを作成
mask_a = torch.zeros((32, 20), dtype=torch.bool, device=device)
mask_a[:, 16:20] = True

plugin_a_model.train()
for step in range(1, 151):
    x, y = get_batch(data, 32, max_seq_len, "code")
    x, y = x.to(device), y.to(device)
    # マスクを渡して新規シナプスのみにルーティングを限定
    logits, _ = plugin_a_model(x, allowed_synapses_mask=mask_a)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    
    opt_a.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(plugin_a_model.parameters(), 1.0)
    opt_a.step()
    
    if step % 50 == 0:
        print(f"Step {step} | CODE Loss: {loss.item():.4f}")

# プラグインA単独でのCODE評価
eval_mask_a = torch.zeros((32, 20), dtype=torch.bool, device=device)
eval_mask_a[:, 16:20] = True
loss_code_independent = eval_domain(plugin_a_model, "code", allowed_mask=eval_mask_a)
print(f"\nPlugin A (Independent) CODE Evaluation Loss: {loss_code_independent:.4f}")

### 4. प्लगइन बी की स्वतंत्र शिक्षा (गणित के लिए)
साथ ही, मान लें कि टीम बी भी उसी बेस मॉडल की प्रतिलिपि बनाती है और समानांतर में एक गणित प्लगइन बनाती है।

In [ ]:
print("=== Training Plugin B (MATH) independently ===")
plugin_b_model = copy.deepcopy(base_model)

# プラグイン用にシナプスを4つ追加し、既存の重みを完全凍結
plugin_b_model.add_synapses(4, freeze_base=True)

opt_b = make_optimizer(plugin_b_model, 5e-4)

# MATHドメインで新しいシナプスのみを使用するためのマスクを作成
mask_b = torch.zeros((32, 20), dtype=torch.bool, device=device)
mask_b[:, 16:20] = True

plugin_b_model.train()
for step in range(1, 151):
    x, y = get_batch(data, 32, max_seq_len, "math")
    x, y = x.to(device), y.to(device)
    logits, _ = plugin_b_model(x, allowed_synapses_mask=mask_b)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    
    opt_b.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(plugin_b_model.parameters(), 1.0)
    opt_b.step()
    
    if step % 50 == 0:
        print(f"Step {step} | MATH Loss: {loss.item():.4f}")

# プラグインB単独でのMATH評価
eval_mask_b = torch.zeros((32, 20), dtype=torch.bool, device=device)
eval_mask_b[:, 16:20] = True
loss_math_independent = eval_domain(plugin_b_model, "math", allowed_mask=eval_mask_b)
print(f"\nPlugin B (Independent) MATH Evaluation Loss: {loss_math_independent:.4f}")

### 5. अल्टीमेट हॉट-स्वैप (टेन्सर्स का भौतिक संयोजन)
अब, यहाँ SRA का जादू है।
टीम ए द्वारा बनाए गए "कोड प्लग-इन" और टीम बी द्वारा बनाए गए "गणित प्लग-इन" को बिना किसी पुनः सीख के बेस मॉडल में संयोजित किया गया है।

विशिष्ट कदम:
1. बेस मॉडल में 8 खाली सिनैप्स जोड़ें (16 + 4 + 4 = 24 सिनैप्स)।
2. अतिरिक्त खाली स्थान पर प्लगइन ए और बी के वेट टेंसर की सीधी मेमोरी कॉपी।

In [ ]:
print("=== Integrating Plugins via Tensor Hot-Swapping ===")

# ベースモデルに8シナプス分の空きソケットを作る
hotswap_model = copy.deepcopy(base_model)
hotswap_model.add_synapses(8, freeze_base=True)

# テンソルの物理コピー
with torch.no_grad():
    for l in range(layers):
        target_block = hotswap_model.blocks[l]
        src_block_a = plugin_a_model.blocks[l]
        src_block_b = plugin_b_model.blocks[l]
        
        # 1. ルーターの埋め込みベクトル(synapse_emb)をコピー
        # plugin_a/b の synapse_emb は新しい4シナプス分のみのテンソル (4, D)
        target_block.router.synapse_emb.data[0:4] = src_block_a.router.synapse_emb.data
        target_block.router.synapse_emb.data[4:8] = src_block_b.router.synapse_emb.data
        
        # 2. Expert (TinySynapse) の重み (w1, w2) をコピー
        target_block.w1.data[0:4] = src_block_a.w1.data
        target_block.w2.data[0:4] = src_block_a.w2.data
        
        target_block.w1.data[4:8] = src_block_b.w1.data
        target_block.w2.data[4:8] = src_block_b.w2.data

print("Tensor Hot-Swapping completed successfully!")

### 6. शून्य-शॉट पूर्ण अलगाव मूल्यांकन
प्रत्येक डोमेन के मेटाडेटा मास्क का उपयोग करके संयुक्त `hotswap_model` पर अनुमान लगाया जाता है।
यह साबित करने के लिए कि हस्तक्षेप पूरी तरह से शून्य है, स्वतंत्र सीखने के दौरान होने वाले नुकसान और संयोजन के बाद होने वाले नुकसान की तुलना करें।

In [ ]:
# 結合モデル用の評価マスクを作成
# TEXT用: ベースシナプス(0~15)のみ許可
mask_text = torch.zeros((32, 24), dtype=torch.bool, device=device)
mask_text[:, 0:16] = True

# CODE用: プラグインAシナプス(16~19)のみ許可
mask_code = torch.zeros((32, 24), dtype=torch.bool, device=device)
mask_code[:, 16:20] = True

# MATH用: プラグインBシナプス(20~23)のみ許可
mask_math = torch.zeros((32, 24), dtype=torch.bool, device=device)
mask_math[:, 20:24] = True

# 結合モデルでのLoss評価
loss_text_hotswap = eval_domain(hotswap_model, "text", allowed_mask=mask_text)
loss_code_hotswap = eval_domain(hotswap_model, "code", allowed_mask=mask_code)
loss_math_hotswap = eval_domain(hotswap_model, "math", allowed_mask=mask_math)

print("\n=== Final Zero-Shot Verification ===")
print(f"TEXT - Base: {loss_text_base:.4f}  | HotSwap: {loss_text_hotswap:.4f}  | Diff: {loss_text_hotswap - loss_text_base:+.4f}")
print(f"CODE - Indep:{loss_code_independent:.4f}  | HotSwap: {loss_code_hotswap:.4f}  | Diff: {loss_code_hotswap - loss_code_independent:+.4f}")
print(f"MATH - Indep:{loss_math_independent:.4f}  | HotSwap: {loss_math_hotswap:.4f}  | Diff: {loss_math_hotswap - loss_math_independent:+.4f}")

# 厳格なアサーションテスト（誤差の確認）
eps = 1e-2
assert abs(loss_text_hotswap - loss_text_base) < eps, "Catastrophic Forgetting detected in TEXT!"
assert abs(loss_code_hotswap - loss_code_independent) < eps, "Interference detected in CODE Plugin!"
assert abs(loss_math_hotswap - loss_math_independent) < eps, "Interference detected in MATH Plugin!"

print("\n🎉 SUCCESS: All plugins were hot-swapped perfectly with ~ZERO interference!")